# Aula 05 — treinamento de cabeças de classificação com Phi-4

O percurso deste notebook é **preparar dados → extrair representações do Phi-4 congelado → treinar as cabeças → calibrar → avaliar → usar as cabeças**. Os dados e scripts estão no GitHub e são baixados na seção 3. Não há um pacote Base64 dentro do notebook.

O classificador retorna suporte, insuficiente ou contradição. A segunda cabeça estima entrega no planeta e retorno com sobrevivência, dado contexto, objetivo e plano. A implementação do agente decide suas ações.

**Para ensinar o treinamento**, execute as seções 1 a 8 em sequência. **Para demonstrar inferência sem esperar o treinamento**, vá à seção 9: sua única célula é independente e usa os pesos já treinados publicados na `main` do GitHub. Não é necessário executar nenhuma célula anterior para essa demonstração.

O dispositivo é escolhido automaticamente: CUDA → MPS/Metal → CPU. Não exige uma GPU específica. O Phi-4 base tem aproximadamente 8,2 GB e é necessário nos dois percursos. A extração dos 816 exemplos é a etapa demorada do treinamento; a demonstração não a repete. Nenhuma entrada é truncada para caber.


## 1. Instalar as dependências

Usaremos PyTorch para executar o Phi-4 congelado e treinar as cabeças. O comando instala os requisitos e conserva uma instalação de PyTorch que já satisfaça a faixa informada.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch>=2.5,<3", "transformers==4.57.6", "numpy>=1.26,<3",
    "safetensors==0.8.0", "huggingface-hub==0.36.2", "certifi>=2024.2.2"])


## 2. Configuração e persistência

Para retomar entre sessões do Colab, monte seu Drive e ajuste `WORK_DIR` para uma pasta persistente, por exemplo `/content/drive/MyDrive/missao-treino`. A montagem do Drive fica a seu critério; o notebook não acessa sua conta automaticamente. `HF_CACHE_DIR` pode apontar para uma pasta persistente de pesos.

`MODEL_DIRECTORY = None` baixa os pesos públicos fixos. Se já os tiver, informe a pasta — os arquivos serão conferidos. `DEVICE="auto"` é o padrão; `"cpu"` força CPU. `PREFILL_CHUNK=None` usa 256 tokens por bloco na CPU e prompt inteiro em aceleradores. Se faltar memória, use `PREFILL_CHUNK=256` em qualquer dispositivo: **o cache conserva todos os tokens anteriores**. Isso altera o custo de execução, não a janela. Não reutilize o mesmo cache de features ao mudar hardware, versão do runtime ou essa configuração; use outro `WORK_DIR`.


In [ ]:
import os
from pathlib import Path
WORK_DIR = Path("missao-espacial-treino")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str((WORK_DIR / "cache-pesos").resolve())
MODEL_DIRECTORY = None
DEVICE = "auto"
TILE_ROWS = 256
PREFILL_CHUNK = None
print("Pasta do treinamento:", WORK_DIR.resolve())


## 3. Baixar os dados e scripts do GitHub

Baixamos `treinamento/piloto-v01/pacote-treinamento.zip` da `main`. O ZIP contém scripts, contratos e dados; não contém os 8,2 GB do Phi-4 base. Conferimos seu SHA-256 antes de abri-lo.

As 600 entradas NLI e as 216 entradas de desfechos ficam separadas da supervisão. A entrada informa o contexto; a supervisão informa o alvo e a partição. Justificativas, testemunhas e IDs reais não são enviados ao modelo.


In [ ]:
import os, json, hashlib, io, zipfile, urllib.request, ssl, certifi
from pathlib import Path
GITHUB_REPO = "AIConnection/missao-espacial"

def baixar(url):
    request = urllib.request.Request(url, headers={"User-Agent": "missao-espacial-aula05"})
    with urllib.request.urlopen(request, timeout=120,
            context=ssl.create_default_context(cafile=certifi.where())) as response:
        return response.read()

commit = json.loads(baixar(f"https://api.github.com/repos/{GITHUB_REPO}/git/ref/heads/main"))["object"]["sha"]
BASE_GITHUB = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{commit}/"
raw = baixar(BASE_GITHUB + "treinamento/piloto-v01/pacote-treinamento.zip")
assert hashlib.sha256(raw).hexdigest() == "0e7714fb0b6c29c762c254418cb4e93fd0d3aa57685b932078ff537b1975e13c", "Pacote incompatível"
PACKAGE_DIR = WORK_DIR / "pacote"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for name in archive.namelist():
        assert (PACKAGE_DIR / name).resolve().is_relative_to(PACKAGE_DIR.resolve()), "Caminho inválido"
    archive.extractall(PACKAGE_DIR)
sys.path.insert(0, str(PACKAGE_DIR / "ferramentas"))
print("Pacote obtido da main, commit:", commit)
from treinar import read_data
inputs, supervision = read_data(PACKAGE_DIR)
print("Exemplos:", len(inputs))


## 4. Conferir o protocolo e executar os testes

NLI: 420 treino, 90 calibração, 90 teste — 200 exemplos de cada classe no total. Episódios, contextos e variantes relacionadas permanecem na mesma partição.

Desfechos: 144 treino, 36 calibração, 36 teste. As famílias estruturais do v04 permanecem separadas. Cada alvo foi estimado com 32 continuações pareadas da engine. As 6.912 continuações **não** são 6.912 exemplos independentes de treino.

A normalização usa apenas treino. São duas heads lineares, 400 épocas fixas, L2 0,1, Adam e learning rate 0,01. A temperatura e o limiar demonstrativo são ajustados na calibração; o teste não participa dessas escolhas.

Os testes usam operações CPU, um Phi-3 minúsculo aleatório e features sintéticas. Validam execução e contratos, **não medem a qualidade do Phi-4**.


In [ ]:
protocol = json.loads((PACKAGE_DIR / "protocolo.json").read_text())
print(json.dumps(protocol, ensure_ascii=False, indent=2))
subprocess.check_call([sys.executable, "-B", str(PACKAGE_DIR / "ferramentas" / "testar.py")])


## 5. Extrair features reais do Phi-4

Esta é a etapa mais cara. O corpo do modelo fica congelado; a feature é o último estado oculto do prompt completo (5.120 dimensões). O prompt da tarefa contém apenas a entrada e as regras constantes. A outra saída desta etapa é a projeção A/B/C do modelo, usada somente como baseline NLI.

Um checkpoint é salvo a cada exemplo. Executar novamente retoma as linhas restantes depois de conferir integridade, ambiente e identidade do modelo. Mantenha o diretório completo. O notebook recusa treinamento se faltarem features. **Não há resultado de qualidade enquanto esta etapa e o treino não terminarem.**


In [ ]:
from treinar import extract
extraction = extract(PACKAGE_DIR, WORK_DIR, MODEL_DIRECTORY, DEVICE, TILE_ROWS, PREFILL_CHUNK)
print("Features concluídas:", len(extraction["rows"]))


## 6. Treinar, calibrar e avaliar

As heads são treinadas na CPU e exportadas com normalização, temperatura e contrato. O relatório mostra as versões crua e calibrada separadamente.

NLI: acurácia, macro F1, matriz de confusão, perda e métricas por classe; comparação com distribuição uniforme e projeção zero-shot A/B/C do Phi-4 (não é a cabeça do Crivo).

Desfechos: Brier e perda Bernoulli calculados sobre as frequências da engine, bins de calibração e comparação com a frequência média do treino. O limiar usa custos unitários de exemplo; ele não determina a intenção do agente. O diagnóstico de planos compara apenas três candidatos e não prova uma estratégia ótima ou generalização universal.


In [ ]:
from treinar import train
result = train(PACKAGE_DIR, WORK_DIR)
nli_test = result["tasks"]["nli"]["evaluation"]["test"]
forecast_test = result["tasks"]["forecast"]["evaluation"]
print("NLI — teste:", json.dumps(nli_test, ensure_ascii=False, indent=2))
for name in ("raw_head", "calibrated_head", "train_prior_baseline"):
    print("Desfechos — teste —", name)
    print(json.dumps(forecast_test[name]["partitions"]["test"], ensure_ascii=False, indent=2))
print("Relatório completo:", WORK_DIR / "resultado.json")


## 7. Inferência experimental

A saída NLI é apenas a classificação. A outra saída são as duas probabilidades dos desfechos, cuja soma é um. A implementação mantém o vínculo com o destinatário, compara alternativas e decide a ação.

A célula abaixo usa um exemplo reservado para demonstração de inferência, sem refazer treinamento ou calibração. A cabeça probabilística permanece condicionada ao amostrador do v04 e aos três programas candidatos. Estes artefatos não substituem automaticamente as heads do simulador ou do Crivo.


In [ ]:
import gc
from inferir import PilotPredictor
predictor = PilotPredictor(PACKAGE_DIR, WORK_DIR / "heads", MODEL_DIRECTORY, DEVICE, TILE_ROWS, PREFILL_CHUNK)
for task in ("nli", "forecast"):
    idx = next(r["index"] for r in supervision if r["task"] == task and r["split"] == "test")
    print(task, predictor.predict(task, inputs[idx]))
del predictor
gc.collect()


## 8. Exportar e interpretar o resultado

O pacote exportado inclui os dados, scripts, contrato do modelo e instruções para rodar localmente. Após o treino real, também inclui heads, previsões reservadas e relatório. Os pesos do corpo Phi-4 e o cache de features não entram no ZIP final. Para retomar a extração entre sessões, preserve o diretório de features separadamente no armazenamento persistente.

Se as heads não superarem os baselines, registre isso e amplie/revise a massa antes de integrar. Este piloto não demonstra cobertura de todos os contextos, perguntas ou estratégias. As heads nunca escolhem a ação do agente.


In [ ]:
export_path = WORK_DIR / "missao-espacial-piloto-exportacao.zip"
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in PACKAGE_DIR.rglob("*"):
        if item.is_file() and "__pycache__" not in item.parts:
            archive.write(item, "pacote/" + str(item.relative_to(PACKAGE_DIR)))
    for item in (WORK_DIR / "heads").rglob("*"):
        if item.is_file():
            archive.write(item, "heads/" + str(item.relative_to(WORK_DIR / "heads")))
    archive.write(WORK_DIR / "resultado.json", "resultado.json")
print("Treinamento exportado:", export_path.resolve())


## 9. Demonstração independente — cabeça treinada do GitHub

**Pode começar diretamente pela célula desta seção, sem executar o treinamento nem as células anteriores.** Ela prepara as dependências, baixa os scripts, dados e pesos treinados da `main` e executa a inferência.

O treinamento utilizou textos do jogo. Agora mantemos os pesos e o processo de inferência e mudamos o assunto dos textos: papelaria, arquivo, alarme, peças e amostras de materiais.

**O que o aluno fará:** ler uma entrada completa, prever sua classe, observar como a entrada é preparada para o Phi-4, executar a cabeça e comparar a classificação com o gabarito.

A tarefa continua a mesma: **a hipótese é sustentada, contradita ou não resolvida pela premissa?** Não estamos perguntando qual ação tomar nem pedindo ao modelo conhecimento externo sobre esses assuntos.

### 9.1. O que é uma entrada?

Um exemplo é um objeto JSON com exatamente dois campos de texto:

| Campo | Pergunta que ele responde | Conteúdo |
|---|---|---|
| `premise` — premissa | O que está disponível para decidir? | Contexto, momento, fatos confirmados, relatos, suposições e regras explícitas. |
| `hypothesis` — hipótese | Qual afirmação será avaliada? | Uma declaração específica sobre esse contexto. |

A hipótese é uma **afirmação**, não precisa ser uma pergunta. Para investigar “há estoque suficiente?”, podemos avaliar a afirmação “um pedido de 14 envelopes pode ser atendido”. Cada hipótese forma um exemplo separado, mesmo quando a premissa é a mesma.

**Os subtítulos dentro da premissa são texto.** O classificador não recebe seis campos estruturados nem uma lista de fatos extraída por outro sistema. Ele recebe as duas strings completas. Os títulos ajudam a distinguir o estatuto de cada informação.

| Rótulo | Relação com a premissa | Cuidado |
|---|---|---|
| suporte | A evidência e as regras sustentam a hipótese. | Não basta parecer plausível. |
| contradição | A evidência e as regras são incompatíveis com a hipótese. | Ausência de confirmação não é negação. |
| insuficiente | A premissa não resolve a hipótese nem sua negação. | Um relato pode provar o que foi dito sem provar o fato relatado. |

A saída pública tem um único campo: `{"classification": "suporte"}`, `{"classification": "insuficiente"}` ou `{"classification": "contradição"}`.


### 9.2. Exemplo guiado: uma premissa, três hipóteses

Esta é a **entrada completa e real** do caso 16 (índice 15 do arquivo). Os caracteres `\n` no JSON representam quebras de linha do texto da premissa.

```json
{
  "premise": "CONTEXTO DISPONÍVEL\nEstoque de papelaria, às 10h. Todas as declarações de estoque se referem a esse instante.\n\nFATOS CONFIRMADOS\nO estoque contém exatamente 14 envelopes. Os envelopes estão lacrados. O pedido atual existe, mas sua quantidade não foi informada.\n\nRELATOS REGISTRADOS — NÃO CONFIRMAM O FATO RELATADO\nNenhum relato adicional.\n\nSUPOSIÇÕES — NÃO SÃO CONFIRMAÇÕES\nNenhuma suposição adicional.\n\nREGRAS EXPLÍCITAS\nUm pedido de envelopes pode ser atendido integralmente se, e somente se, a quantidade pedida for menor ou igual ao estoque disponível.",
  "hypothesis": "Um pedido de 14 envelopes pode ser atendido integralmente às 10h."
}
```

Para decidir, identificamos **14 envelopes confirmados**, a regra **quantidade pedida ≤ estoque** e a hipótese **pedido de 14**. Como **14 ≤ 14**, a classe esperada é **suporte**.

Mantendo exatamente a mesma premissa, os arquivos contêm três exemplos:

| Caso no notebook | Hipótese enviada | Raciocínio para o gabarito | Classe esperada |
|---|---|---|---|
| 16 | Um pedido de 14 envelopes pode ser atendido integralmente às 10h. | 14 ≤ 14: a regra permite atender. | suporte |
| 19 | Um pedido de 15 envelopes pode ser atendido integralmente às 10h. | 15 > 14: a regra impede atender. | contradição |
| 17 | O pedido atual pode ser atendido integralmente às 10h. | O pedido existe, mas sua quantidade é desconhecida. Não podemos aplicar a comparação. | insuficiente |

**Distinção essencial:** a premissa não mudou; mudou a afirmação avaliada. Um contexto rico pode continuar insuficiente para uma hipótese específica. O gabarito acima é uma explicação para o aluno e **não entra no modelo**.


### 9.3. Exemplo contrastante: “foi dito” versus “é fato”

Leia a premissa completa desta segunda família:

```text
CONTEXTO DISPONÍVEL
Análise de uma mensagem sobre um alarme, às 9h.

FATOS CONFIRMADOS
A mensagem abaixo foi registrada integralmente às 9h. A porta da sala está confirmadamente fechada.

RELATOS REGISTRADOS — NÃO CONFIRMAM O FATO RELATADO
Há um único relato registrado: «O alarme está acionado às 9h». Não há medição confirmada do estado do alarme.

SUPOSIÇÕES — NÃO SÃO CONFIRMAÇÕES
Nenhuma suposição adicional.

REGRAS EXPLÍCITAS
Nenhuma regra adicional.
```

| Caso | Hipótese enviada | Classe esperada | Por quê? |
|---|---|---|---|
| 12 | O relato registrado afirma que o alarme está acionado às 9h. | suporte | A hipótese descreve o conteúdo registrado, não o estado real do alarme. |
| 14 | O alarme está realmente acionado às 9h. | insuficiente | O registro comprova o relato; não confirma seu conteúdo como fato. |
| 25 | O único relato registrado afirma que o alarme está desativado às 9h. | contradição | O conteúdo integral do único relato é acionado, e não desativado. |

A mensagem registrada confirma a existência e o conteúdo do relato. Não há medição do alarme. Por isso, **o relato foi registrado** pode receber suporte, enquanto **o alarme está realmente acionado** recebe insuficiente. Não é uma regra “todo relato = insuficiente”: depende do que a hipótese afirma.

**Preveja antes de executar:** se uma medição confirmada às 9h dissesse que o alarme está desativado, qual seria a classe da hipótese sobre o estado real? Explique qual evidência mudou a decisão.


### 9.4. Da entrada do caso à entrada efetiva do Phi-4

Há duas etapas diferentes. O notebook fornece o par; o carregador prepara o prompt exatamente como no treinamento:

```text
Par do caso: {premise, hypothesis}
                 ↓
Premissa efetiva = prefixo fixo + duas quebras de linha + premise
Hipótese efetiva = hypothesis, sem alteração
                 ↓
Mensagens: instrução system + JSON do par efetivo na mensagem user
                 ↓
Chat template do Phi-4 + tokenização, sem truncamento
                 ↓
Phi-4 congelado → representação de 5.120 valores
                 ↓
Normalização e cabeça treinadas → classification
```

**O prefixo também é parte da entrada do modelo.** Ele contém regras sobre evidência e regras específicas do jogo. Mantemos todas porque a cabeça foi treinada com esse prefixo. Portanto, o experimento avalia novos assuntos **sob o protocolo original**, não um classificador sem qualquer referência ao jogo.

A instrução `system` original usa A/B/C. Ela foi utilizada para extrair as representações durante o treinamento e permanece por compatibilidade. Nesta inferência, **não pedimos ao modelo que gere uma letra**: a cabeça aplica seus pesos à representação e retorna uma das três classes.

A célula independente abaixo mostra a instrução, o prefixo e o prompt serializado do primeiro caso, incluindo sua quantidade de tokens. Depois mostra a entrada completa e a saída de cada inferência.


### 9.5. Como estão separados os 30 casos?

| Família | Base | Variantes | Total | O que exercita |
|---|---:|---:|---:|---|
| Estoque | 3 | 1 | 4 | Regra explícita e quantidade do pedido desconhecida. |
| Fronteira numérica | 3 | 1 | 4 | `> 14`, `≥ 14` e quantidade futura desconhecida. |
| Tempo | 3 | 1 | 4 | Leitura no momento correto e ausência de previsão futura. |
| Relato | 3 | 1 | 4 | Conteúdo registrado versus estado real do ambiente. |
| Relatos conflitantes | 3 | 0 | 3 | Divergência entre relatos não resolve o fato. |
| Suposição | 3 | 1 | 4 | Separar hipótese não verificada de observação confirmada. |
| Categorias abertas | 3 | 1 | 4 | A presença de cobre não exclui a presença de ferro. |
| Regra explícita | 3 | 0 | 3 | Aplicar condições declaradas; não inventar evidência de assinatura. |
| **Total** | **24** | **6** | **30** | **10 por classe.** |

Os seis exemplos variantes modificam redação e acrescentam informação irrelevante. São relacionados aos exemplos de base; não equivalem a seis observações independentes novas. Os rótulos foram definidos antes da inferência. Esta amostra não cobre todos os assuntos ou formas de escrever.

| Arquivo | Uso | Entra no prompt? |
|---|---|---|
| `inputs.jsonl` | Os 30 objetos com premissa e hipótese. | Sim, um objeto por inferência, preparado com o prefixo. |
| `supervisao.jsonl` | Classe esperada, justificativa, família, índice e variante. | **Não.** Serve para comparar e interpretar depois. |
| `protocolo.json` | Distribuição, limites do experimento e hashes dos dados. | **Não.** Serve para validar a avaliação. |
| `nli.npz` e `artefato.json` | Pesos, normalização, temperatura e contrato da cabeça. | Não são texto do prompt; configuram a inferência. |

O índice do caso organiza a avaliação. Não é uma identidade de agente e não é passado ao modelo. A cabeça de desfechos não é testada fora da missão: entrega e retorno têm significados específicos do jogo.

**Origem dos arquivos:** resolvemos o commit atual da `main` e baixamos pesos e casos desse mesmo commit. Os hashes são conferidos. O corpo Phi-4 é obtido da revisão fixa do pacote; as cabeças já treinadas vêm do GitHub. Nenhum treino ou ajuste de temperatura ocorre aqui.


### 9.6. Execute esta célula para demonstrar inferência

Ela usa **a cabeça publicada no GitHub**, independentemente de um treinamento feito nesta sessão. Executa os 30 casos definidos anteriormente, sem treinar ou recalibrar. Para uma demonstração curta, substitua `INDICES = list(range(30))` por `INDICES = [15, 18, 16, 11, 13, 24]`: são os seis exemplos guiados de estoque e relato; os índices começam em zero.

Cada resultado aparece nesta ordem: **premissa → hipótese → saída do modelo → gabarito e justificativa**. O gabarito só é usado depois de `predict`, na comparação. Apenas o par entra no carregador.

Acurácia é a fração de acertos; precisão mede a correção das previsões de uma classe; recall mede quantos casos dessa classe foram reconhecidos; macro F1 dá igual peso às três classes. Na matriz de confusão, as linhas são as classes esperadas e as colunas são as classes previstas.

A execução local histórica acertou 23/30. A célula abaixo realiza inferências novas; não lê previsões salvas. Resultados podem variar com o dispositivo. Erros fora do domínio delimitam a generalização observada; esta amostra não permite concluir sua causa. A combinação de paráfrase e informação irrelevante nas variantes também não isola qual mudança teve efeito.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch>=2.5,<3", "transformers==4.57.6", "numpy>=1.26,<3",
    "safetensors==0.8.0", "huggingface-hub==0.36.2", "certifi>=2024.2.2"])
import gc, time, numpy as np
from pathlib import Path
WORK_DIR = Path("missao-espacial-demonstracao")
WORK_DIR.mkdir(parents=True, exist_ok=True)
import os, json, hashlib, io, zipfile, urllib.request, ssl, certifi
from pathlib import Path
GITHUB_REPO = "AIConnection/missao-espacial"

def baixar(url):
    request = urllib.request.Request(url, headers={"User-Agent": "missao-espacial-aula05"})
    with urllib.request.urlopen(request, timeout=120,
            context=ssl.create_default_context(cafile=certifi.where())) as response:
        return response.read()

commit = json.loads(baixar(f"https://api.github.com/repos/{GITHUB_REPO}/git/ref/heads/main"))["object"]["sha"]
BASE_GITHUB = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{commit}/"
raw = baixar(BASE_GITHUB + "treinamento/piloto-v01/pacote-treinamento.zip")
assert hashlib.sha256(raw).hexdigest() == "0e7714fb0b6c29c762c254418cb4e93fd0d3aa57685b932078ff537b1975e13c", "Pacote incompatível"
PACKAGE_DIR = WORK_DIR / "pacote"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for name in archive.namelist():
        assert (PACKAGE_DIR / name).resolve().is_relative_to(PACKAGE_DIR.resolve()), "Caminho inválido"
    archive.extractall(PACKAGE_DIR)
sys.path.insert(0, str(PACKAGE_DIR / "ferramentas"))
print("Pacote obtido da main, commit:", commit)

os.environ["HF_HOME"] = str((WORK_DIR / "cache-pesos").resolve())
HEADS_DIR = WORK_DIR / "heads-github"
HEADS_DIR.mkdir(parents=True, exist_ok=True)
head_path = "treinamento/piloto-v01/execucao-local-01/heads/"
artifact = json.loads(baixar(BASE_GITHUB + head_path + "artefato.json"))
for task in ("nli", "forecast"):
    content = baixar(BASE_GITHUB + head_path + task + ".npz")
    assert hashlib.sha256(content).hexdigest() == artifact["heads"][task], "Peso corrompido"
    (HEADS_DIR / (task + ".npz")).write_bytes(content)
(HEADS_DIR / "artefato.json").write_text(json.dumps(artifact))
case_path = "treinamento/piloto-v01/avaliacao-fora-dominio-01/"
protocol = json.loads(baixar(BASE_GITHUB + case_path + "protocolo.json"))
for name in ("inputs.jsonl", "supervisao.jsonl"):
    content = baixar(BASE_GITHUB + case_path + name)
    assert hashlib.sha256(content).hexdigest() == protocol["data_sha256"][name], "Dados corrompidos"
    (WORK_DIR / name).write_bytes(content)
pares = [json.loads(line) for line in (WORK_DIR / "inputs.jsonl").read_text().splitlines()]
gabaritos = [json.loads(line) for line in (WORK_DIR / "supervisao.jsonl").read_text().splitlines()]
INDICES = list(range(30))

from inferir import PilotPredictor
from phi4_portatil import NLI_SYSTEM
predictor = PilotPredictor(PACKAGE_DIR, HEADS_DIR, device="auto")
primeiro = pares[INDICES[0]]
par_efetivo = {"premise": predictor.prefix + "\n\n" + primeiro["premise"], "hypothesis": primeiro["hypothesis"]}
prompt = predictor.backend.tokenizer.apply_chat_template(
    [{"role": "system", "content": NLI_SYSTEM},
     {"role": "user", "content": json.dumps(par_efetivo, ensure_ascii=False)}],
    tokenize=False, add_generation_prompt=True)
print("PROMPT COMPLETO DO PRIMEIRO CASO:\n" + prompt)
print("Tokens:", len(predictor.backend.encode_nli(primeiro, predictor.prefix)))
print("Dispositivo:", predictor.backend.identity["device"])

resultados = []
for index in INDICES:
    par = pares[index]
    print(f"\nCASO {index + 1:02d}")
    print("PREMISSA:\n" + par["premise"])
    print("HIPÓTESE: " + par["hypothesis"])
    inicio = time.perf_counter()
    saida = predictor.predict("nli", par)
    print("SAÍDA DO MODELO:", saida)
    alvo = gabaritos[index]
    print("GABARITO (fora do modelo):", alvo["expected"])
    print("JUSTIFICATIVA:", alvo["rationale"])
    resultados.append({"index": index, "expected": alvo["expected"], "output": saida,
                       "seconds": time.perf_counter() - inicio})
del predictor
gc.collect()

classes = ["suporte", "insuficiente", "contradição"]
matriz = np.zeros((3, 3), dtype=int)
for linha in resultados:
    matriz[classes.index(linha["expected"]), classes.index(linha["output"]["classification"])] += 1
metricas_classe = {}
for i, classe in enumerate(classes):
    tp = int(matriz[i, i])
    previstos = int(matriz[:, i].sum())
    reais = int(matriz[i, :].sum())
    precisao = tp / max(previstos, 1)
    recall = tp / max(reais, 1)
    f1 = 2 * tp / max(previstos + reais, 1)
    metricas_classe[classe] = {"precision": precisao, "recall": recall, "f1": f1, "count": reais}
relatorio = {"github_commit": commit, "heads_sha256": artifact["heads"], "indices": INDICES,
             "accuracy": float(np.trace(matriz) / matriz.sum()),
             "macro_f1": sum(m["f1"] for m in metricas_classe.values()) / 3,
             "labels": classes, "confusion": matriz.tolist(), "per_class": metricas_classe}
print("\nMÉTRICAS DESTA EXECUÇÃO:\n" + json.dumps(relatorio, ensure_ascii=False, indent=2))
# Cada execução produz um arquivo novo; não substitui as avaliações anteriores.
from datetime import datetime, timezone
execucao = WORK_DIR / ("resultado-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
execucao.write_text(json.dumps({"report": relatorio, "predictions": resultados}, ensure_ascii=False, indent=2))
print("Resultados salvos:", execucao.resolve())
